In [1]:
import numpy as np
import statsmodels.api as sm
import linearmodels as lm
import time
import pyensmallen as pe

np.random.seed(94305)

from gmm.gmm import GMMEstimator  # noqa: E402


Single endogeneous variable and single instrument DGP with varying instrument strength ($\pi$) and degree of endogeneity($\rho$).


In [2]:
def dgp(n = 1000,
        beta = np.array([-0.5, 1.2]),
        rho = 0.7,
        pi = np.array([0.5, -0.1])):
    ε = np.random.normal(0, 1, n)
    z = np.random.normal(0, 1, n * pi.shape[0]).reshape(n, pi.shape[0])
    # Generate endogenous x, influenced by the instrument
    x = z @ pi + ε * rho + np.random.normal(0, 1, n)
    X = np.c_[np.ones(n), x]
    # heteroskedasticity
    y = X @ beta + ε + (X[:, 1] > 0) * np.random.normal(0, 1, n)
    return y, X, z

# No Endogeneity 

OLS and IV with X as its own instrument should produce the same estimates.

In [3]:
y, X, z = dgp(pi = np.array([0]), rho = 0)
print(sm.OLS(y, X).fit(cov_type = "HC2").summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.4976      0.038    -13.249      0.000      -0.571      -0.424
x1             1.1822      0.036     32.849      0.000       1.112       1.253


### GMM using Scipy Minimization

In [4]:
%%time
ψ = lambda z, y, x, beta: z * (y - x @ beta)[:, np.newaxis]
gmm_scipy = GMMEstimator(ψ)
gmm_scipy.fit(np.c_[np.ones(z.shape[0]), X[:, 1]], y, X)
gmm_scipy.summary()

CPU times: user 21.5 ms, sys: 0 ns, total: 21.5 ms
Wall time: 7.89 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4976,0.0,-13261.8743,0.0,-0.4977,-0.4975
1,1.1822,0.0,32909.9285,0.0,1.1821,1.1823


### GMM using `pyensmallen`

In [5]:
%%time
ψ = lambda z, y, x, beta: z * (y - x @ beta)[:, np.newaxis]
gmm_pe = GMMEstimator(ψ, backend = "pyensmallen")
gmm_pe.fit(np.c_[np.ones(z.shape[0]), X[:, 1]], y, X)
gmm_pe.summary()

CPU times: user 1.66 ms, sys: 1.98 ms, total: 3.64 ms
Wall time: 3.3 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4976,0.0,-13261.9715,0.0,-0.4977,-0.4975
1,1.1822,0.0,32909.8360,0.0,1.1821,1.1823


In [6]:
%%time
ψ = lambda z, y, x, beta: z * (y - x @ beta)[:, np.newaxis]
gmm_pe = GMMEstimator(ψ, backend = "pyensmallen")
gmm_pe.fit(np.c_[np.ones(z.shape[0]), X[:, 1]], y, X)
gmm_pe.summary()

CPU times: user 2.73 ms, sys: 0 ns, total: 2.73 ms
Wall time: 2.41 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4976,0.0,-13261.9708,0.0,-0.4977,-0.4975
1,1.1822,0.0,32909.8371,0.0,1.1821,1.1823


### GMM using Torch Minimization

In [6]:
%%time
def moment_cond(z, y, x, beta):
    residuals = (y - x @ beta).unsqueeze(-1)
    return z * residuals

gmm = GMMEstimator(moment_cond, backend  = "torch")
gmm.fit(np.c_[np.ones(z.shape[0]), X[:, 1]], y, X)
gmm.summary()

CPU times: user 944 ms, sys: 53.8 ms, total: 998 ms
Wall time: 730 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4976,0.0,-13261.6551,0.0,-0.4977,-0.4975
1,1.1822,0.0,32909.8149,0.0,1.1821,1.1823


Faster optimizer (default argument is limited-memory BFGS, but BFGS works faster for small problems)

In [7]:
%%time
def moment_cond(z, y, x, beta):
    residuals = (y - x @ beta).unsqueeze(-1)
    return z * residuals

gmm = GMMEstimator(moment_cond, backend = "torch")
gmm.fit(np.c_[np.ones(z.shape[0]), X[:, 1]], y, X, fit_method='bfgs')
gmm.summary()

CPU times: user 874 ms, sys: 89.3 ms, total: 963 ms
Wall time: 82.3 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4994,0.0039,-128.5068,0.0,-0.5070,-0.4918
1,1.1953,0.0039,308.5945,0.0,1.1877,1.2029


# With Endogeneity 

Over-identified: 2 instruments and 1 endogenous variable.

OLS is inconsistent. Also confirm `GMMEstimator` returns the same answer as IV2SLS.

In [7]:
y, X, z = dgp()
print(sm.OLS(y, X).fit().summary().tables[1])

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.4981      0.003   -143.251      0.000      -0.505      -0.491
x1             1.5971      0.003    608.599      0.000       1.592       1.602


In [8]:
%%time
ψ = lambda z, y, x, beta: z * (y - x @ beta)[:, np.newaxis]
gmm = GMMEstimator(ψ)
gmm.fit(np.c_[np.ones(z.shape[0]), z], y, X)
gmm.summary()

CPU times: user 190 ms, sys: 6.79 ms, total: 197 ms
Wall time: 155 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4983,0.0039,-129.0528,0.0,-0.5059,-0.4908
1,1.1964,0.0076,157.7118,0.0,1.1815,1.2113


In [9]:
%%time
def moment_cond(z, y, x, beta):
    residuals = (y - x @ beta).unsqueeze(-1)
    return z * residuals

gmm = GMMEstimator(moment_cond, backend = "torch")
gmm.fit(np.c_[np.ones(z.shape[0]), z], y, X)
gmm.summary()

CPU times: user 263 ms, sys: 23.2 ms, total: 286 ms
Wall time: 88 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4983,0.0039,-129.0516,0.0,-0.5059,-0.4907
1,1.1964,0.0076,157.7132,0.0,1.1815,1.2113


Faster optimizer in torch

In [10]:
%%time
def moment_cond(z, y, x, beta):
    residuals = (y - x @ beta).unsqueeze(-1)
    return z * residuals

gmm = GMMEstimator(moment_cond, backend = "torch")
gmm.fit(np.c_[np.ones(z.shape[0]), z], y, X, fit_method='bfgs')
gmm.summary()

CPU times: user 290 ms, sys: 15.7 ms, total: 306 ms
Wall time: 83.6 ms


,coef,std err,t,p-value,[0.025,0.975]
0,-0.4983,0.0039,-129.0435,0.0,-0.5058,-0.4907
1,1.1964,0.0076,157.7158,0.0,1.1816,1.2113


Confirm with linearmodels

In [11]:
lm.iv.model.IV2SLS(y, None, X, np.c_[np.ones(z.shape[0]), z]).fit().summary.tables[1]

/home/alal/Desktop/forks/linearmodels/linearmodels/shared/linalg.py:64: RuntimeWarning: divide by zero encountered in divide
  return vecs @ np.diag(1 / np.sqrt(vals)) @ vecs.T
/home/alal/Desktop/forks/linearmodels/linearmodels/shared/linalg.py:64: RuntimeWarning: invalid value encountered in matmul
  return vecs @ np.diag(1 / np.sqrt(vals)) @ vecs.T
/home/alal/Desktop/forks/linearmodels/linearmodels/iv/model.py:605: RuntimeWarning: invalid value encountered in matmul
  q = vpmzv_sqinv @ (ex1.T @ ex1) @ vpmzv_sqinv


,Parameter,Std. Err.,T-stat,P-value,Lower CI,Upper CI
endog.0,-0.4983,0.0039,-129.06,0.0000,-0.5059,-0.4907
endog.1,1.1965,0.0076,157.72,0.0000,1.1816,1.2113


Identical estimates and standard errors.

### GMM using PyEnsmallen Minimization

Now let's test the PyEnsmallen backend which should offer improved performance for optimization problems.

In [ ]:
%%time
if pyensmallen_available:
    ψ = lambda z, y, x, beta: z * (y - x @ beta)[:, np.newaxis]
    gmm_pe = GMMEstimator(ψ, backend="pyensmallen")
    gmm_pe.fit(np.c_[np.ones(z.shape[0]), z], y, X)
    gmm_pe.summary()
else:
    print("PyEnsmallen is not installed. Install with: pip install pyensmallen")

## Performance Comparison of Different Backends

Let's compare the performance of all backends on the same problem.

In [ ]:
def benchmark_backend(backend, n_runs=5):
    """Run GMM estimation with a specific backend multiple times and measure average execution time"""
    times = []

    # Define moment condition function based on backend
    if backend == "torch":
        def moment_cond(z, y, x, beta):
            residuals = (y - x @ beta).unsqueeze(-1)
            return z * residuals
    else:
        moment_cond = lambda z, y, x, beta: z * (y - x @ beta)[:, np.newaxis]

    # Run estimation multiple times
    for _ in range(n_runs):
        gmm = GMMEstimator(moment_cond, backend=backend)

        start_time = time.time()
        gmm.fit(np.c_[np.ones(z.shape[0]), z], y, X, verbose=False)
        end_time = time.time()

        times.append(end_time - start_time)

    return np.mean(times)

In [ ]:
import matplotlib.pyplot as plt

# List of backends to benchmark
backends = ["scipy", "torch"]
if pyensmallen_available:
    backends.append("pyensmallen")

# Run benchmarks
benchmark_results = {}
for backend in backends:
    avg_time = benchmark_backend(backend)
    benchmark_results[backend] = avg_time
    print(f"{backend.capitalize()} average time: {avg_time:.4f} seconds")

In [ ]:
# Plot benchmark results
plt.figure(figsize=(10, 5))
plt.bar(benchmark_results.keys(), benchmark_results.values())
plt.ylabel('Average Time (seconds)')
plt.title('Performance Comparison of GMM Backends')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)

# Add labels on top of bars
for i, (k, v) in enumerate(benchmark_results.items()):
    plt.text(i, v + 0.02, f"{v:.4f}s", ha='center')

plt.tight_layout()
plt.show()

## Comparing Parameter Estimates Across Backends

Let's verify that all backends produce comparable parameter estimates.

In [ ]:
import pandas as pd

# Get parameter estimates from different backends
scipy_moment = lambda z, y, x, beta: z * (y - x @ beta)[:, np.newaxis]
gmm_scipy = GMMEstimator(scipy_moment, backend="scipy")
gmm_scipy.fit(np.c_[np.ones(z.shape[0]), z], y, X, verbose=False)

torch_moment = lambda z, y, x, beta: z * (y - x @ beta).unsqueeze(-1)
gmm_torch = GMMEstimator(torch_moment, backend="torch")
gmm_torch.fit(np.c_[np.ones(z.shape[0]), z], y, X, verbose=False)

# Compare parameters
results_dict = {
    "True Value": np.array([-0.5, 1.2]),  # True parameter values from DGP
    "SciPy": gmm_scipy.theta_,
    "PyTorch": gmm_torch.theta_,
}

if pyensmallen_available:
    gmm_pe = GMMEstimator(scipy_moment, backend="pyensmallen")
    gmm_pe.fit(np.c_[np.ones(z.shape[0]), z], y, X, verbose=False)
    results_dict["PyEnsmallen"] = gmm_pe.theta_

# Create comparison dataframe
comparison_df = pd.DataFrame(results_dict, index=["Intercept", "Slope"])
comparison_df

In [ ]:
# Calculate absolute error relative to true values
for backend in list(results_dict.keys())[1:]:
    comparison_df[f"{backend} Error"] = np.abs(comparison_df[backend] - comparison_df["True Value"])

# Display formatted results
comparison_df.round(4)